# Silver Layer — Caregiver (ICU)

## Source
`cdr_mimiciv.bronze.caregiver`

## Target
`cdr_mimiciv.silver.caregiver`

The target table is published with CREATE OR REPLACE semantics; downstream Gold depends on it.

## Grain
One row represents one unique caregiver identified by `caregiver_id`.

## Source-file evidence
Identifiers are conformed to the documented STRING type with an explicit cast. Bronze CSV inference loaded this release as INT; the cast preserves its identifier values.

## Silver objective
Create a reusable and clinically interpretable caregiver entity.

## Engineering principles
- Preserve source clinical meaning.
- Do not impute missing clinical values.
- Do not remove records unless a validated rule requires it.
- Apply only deterministic, reusable Silver transformations.
- Type note: the caregiver documentation types `caregiver_id` as `VARCHAR(10)`, whereas the ICU event tables type it as `INTEGER`. This table keeps the documented string type.

In [0]:
from pyspark.sql import functions as F
import re

dbutils.widgets.text("source_schema", "cdr_mimiciv.bronze")
dbutils.widgets.text("target_schema", "cdr_mimiciv.silver")

BRONZE = dbutils.widgets.get("source_schema").strip()
SILVER = dbutils.widgets.get("target_schema").strip()

for _schema in (BRONZE, SILVER):
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*\.[A-Za-z_][A-Za-z0-9_]*", _schema):
        raise ValueError(f"Schema must be catalog.schema: {_schema}")
if BRONZE == SILVER:
    raise ValueError("source_schema and target_schema must differ")

TARGET = f"{SILVER}.caregiver"

In [0]:
caregiver_bronze = spark.table(f"{BRONZE}.caregiver")

display(caregiver_bronze.limit(20))

caregiver_id
3
4
6
9
12
16
41
45
48
54


In [0]:
caregiver_profile = caregiver_bronze.select(
    F.count("*").alias("total_rows"),
    F.countDistinct("caregiver_id").alias("unique_caregiver_ids"),
    F.sum(
        F.when(F.col("caregiver_id").isNull(), 1).otherwise(0)
    ).alias("null_caregiver_ids")
)

display(caregiver_profile)

total_rows,unique_caregiver_ids,null_caregiver_ids
17984,17984,0


In [0]:
duplicate_caregivers = (
    caregiver_bronze
    .groupBy("caregiver_id")
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate caregiver_ids:", duplicate_caregivers.count())

Duplicate caregiver_ids: 0


In [0]:
# check missing values
display(
    caregiver_bronze.select(
        *[
            F.sum(
                F.when(F.col(c).isNull(), 1).otherwise(0)
            ).alias(c)
            for c in [
                "caregiver_id"
            ]
        ]
    )
)

caregiver_id
0


In [0]:
# inspect rescued data (only if the Bronze loader created the column)
if "_rescued_data" in caregiver_bronze.columns:
    rescued_count = (
        caregiver_bronze
        .filter(F.col("_rescued_data").isNotNull())
        .count()
    )
    print("Rows with rescued data:", rescued_count)
else:
    print("No _rescued_data column in Bronze table; skipped.")

No _rescued_data column in Bronze table; skipped.


In [0]:
display(
    caregiver_bronze.select(
        F.max(F.length(F.col("caregiver_id").cast("string"))).alias("max_id_length")
    )
)

max_id_length
6


In [0]:
caregiver_silver = (
    caregiver_bronze
    .select(
        F.col("caregiver_id")
            .cast("string")
            .alias("caregiver_id")
    )

    # Real technical processing timestamp
    .withColumn(
        "_silver_processed_at",
        F.current_timestamp()
    )
)

In [0]:
# pre-publish validation
bronze_rows = caregiver_bronze.count()
pre = caregiver_silver.select(
    F.count("*").alias("silver_rows"),
    F.countDistinct("caregiver_id").alias("unique_ids"),
    F.sum(F.when(F.col("caregiver_id").isNull(), 1).otherwise(0)).alias("null_ids"),
    F.sum(F.when(F.col("_silver_processed_at").isNull(), 1).otherwise(0)).alias("null_timestamp")
).first()
display(caregiver_silver.limit(5))

assert pre["silver_rows"] == bronze_rows, "Row count differs from Bronze"
assert pre["null_ids"] == 0, "NULL caregiver_id present"
assert pre["unique_ids"] == pre["silver_rows"], "caregiver_id is not unique"
assert pre["null_timestamp"] == 0, "NULL processing timestamp present"

caregiver_id,_silver_processed_at
3,2026-10-19T22:48:59.371Z
4,2026-10-19T22:48:59.371Z
6,2026-10-19T22:48:59.371Z
9,2026-10-19T22:48:59.371Z
12,2026-10-19T22:48:59.371Z


In [0]:
# publish (CREATE OR REPLACE; downstream Gold depends on this table)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER}")
caregiver_silver.writeTo(TARGET).using("delta").createOrReplace()

print(f"Published: {TARGET}")

Published: cdr_mimiciv.silver.caregiver


In [0]:
# post-publish validation of the published table
EXPECTED_SCHEMA = {
    "caregiver_id": "string",
    "_silver_processed_at": "timestamp",
}

published = spark.table(TARGET)
actual_schema = {f.name: f.dataType.simpleString() for f in published.schema.fields}
assert list(actual_schema.items()) == list(EXPECTED_SCHEMA.items()), (
    f"Schema mismatch: {actual_schema}"
)

post = published.select(
    F.count("*").alias("rows"),
    F.countDistinct("caregiver_id").alias("unique_ids"),
    F.sum(F.when(F.col("caregiver_id").isNull(), 1).otherwise(0)).alias("null_ids")
).first()

assert post["rows"] == bronze_rows, "Published row count differs from Bronze"
assert post["null_ids"] == 0
assert post["unique_ids"] == post["rows"]
print("Validated:", post.asDict())

Validated: {'rows': 17984, 'unique_ids': 17984, 'null_ids': 0}


In [0]:
dbutils.notebook.exit("caregiver:OK")